# Redis 

In [1]:
import redis

# Connect to Redis
r = redis.Redis(
    host='localhost',
    port=6379,
    decode_responses=True  # This automatically decodes bytes to strings
)


In [ ]:
# Adding a list to redis set
match_ids = [123,432,999,850]
r.sadd('new_matches', *match_ids)
r.smembers('new_matches')

# Redis Stream

Create a stream called "status_changes" and add three events for a match transitioning from "started" to "ongoing" to "predicted". Then read all events from the stream.

In [ ]:
from datetime import datetime as dt
import time

r.xadd('status_changes', {
    'match_id':'729',
    'status':'started',
    'timestamp':dt.now().strftime("%Y-%m-%d %H:%M:%S")
})

time.sleep(1)

r.xadd('status_changes', {
    'match_id':'729',
    'status':'ongoing',
    'timestamp':dt.now().strftime("%Y-%m-%d %H:%M:%S")
})

time.sleep(1)

r.xadd('status_changes', {
    'match_id':'729',
    'status':'predicted',
    'timestamp':dt.now().strftime("%Y-%m-%d %H:%M:%S")
})


In [ ]:
events = r.xread({'status_changes': '0'})
for stream_name, stream_events in events:
    for event_id, data in stream_events:
        print(f"Event {event_id} : {data}")
        

### Consumer Stream

In [ ]:
pending = r.xpending('status_changes', 'consumer_group')
print(f"Pending messages: {pending}")

In [ ]:
try:
    r.xgroup_create('status_changes', 'consumer_group', id='0', mkstream=True)
except redis.exceptions.ResponseError as e:
    if 'BUSYGROUP' in str(e):
        print("Group already exists")
    else:
        raise e
    
# Read new messages as a specific consumer
events = r.xreadgroup('consumer_group', 'consumer1', 
                     {'status_changes': '>'}, count=10)  # > is a special Id for new messages. '0' is starting from the first pending messages. 


print(f"Events received: {events}")

# for stream_name, stream_events in events:
#     for event_id, data in stream_events:
#         print(f"Processing {data['match_id']} with status {data['status']}")
#         # Process the event...
        
#         # Acknowledge processing
#         r.xack('match_events', 'prediction_group', event_id)

In [ ]:
pending = r.xpending('status_changes', 'consumer_group')
pending_detailed = r.xpending_range('status_changes', 'consumer_group', '-', '+', 10)
# -: smallest possible +: largest possible 10: number of pending to return
print(pending_detailed)

In [ ]:
if pending_detailed:
    r.xclaim('status_changes', 'consumer_group', 'consumer2',
            min_idle_time=60000,  # 60 seconds
            message_ids=[pending_detailed[0]['message_id']])

In [ ]:
r.xpending('status_changes', 'consumer_group')

#### Exercise
Create a consumer group for a "prediction_requests" stream. Add a match event, consume it with one consumer, then check for any pending messages.

In [ ]:
from datetime import datetime as dt

r.xadd('prediction_requests', {
    'match_id':'150',
    'status': 'started',
    'timestamp':dt.now().strftime('%Y-%m-%d %H:%M:%S')
})

In [ ]:
try:
    r.xgroup_create('prediction_requests', 'predict_group', id='0', mkstream=True)
except redis.exceptions.ResponseError as e:
    if 'BUSYGROUP' in str(e):
        print("Group already exists")
    else:
        raise e

event = r.xreadgroup('predict_group', 'consumer1', {'prediction_requests':'>'})
event


In [ ]:
r.xpending('prediction_requests','predict_group')

### Stream Management

In [ ]:
r.xinfo_stream('status_changes')

In [ ]:
groups = r.xinfo_groups('status_changes')
groups

In [ ]:
test_stream = 'test_trim_stream'
for i in range(10):
    r.xadd(test_stream, {'index':str(i)})
    
stream_info = r.xinfo_stream(test_stream)
stream_info

In [ ]:
r.xtrim(test_stream, maxlen=5, approximate=False) # For large amount of messages to trim, use approximate=True (Default). 
# If you need exact value, use approximate=False
after_info = r.xinfo_stream(test_stream)
after_info

## Real World Pattern for Match Prediction

In [1]:
from src.fetch_data.fetch_live_leagues import retrieve_live_league_games

In [2]:
def print_games():
    games = retrieve_live_league_games()
    for game in games:
        print('game', game)
        
print_games()

game {'match_id': 8237708401, 'radiant_team_id': 8291895, 'radiant_name': 'Tundra Esports', 'dire_team_id': 1838315, 'dire_name': 'Team Secret', 'game_duration': 0, 'start_time': 1743506030.285075, 'radiant_win': -1, '0_account_id': 86698277, '0_hero_id': 0, '1_account_id': 136829091, '1_hero_id': 0, '2_account_id': 127617979, '2_hero_id': 0, '3_account_id': 93618577, '3_hero_id': 0, '4_account_id': 103735745, '4_hero_id': 0, '128_account_id': 1171243748, '128_hero_id': 0, '129_account_id': 58513047, '129_hero_id': 0, '130_account_id': 374875067, '130_hero_id': 0, '131_account_id': 105045291, '131_hero_id': 0, '132_account_id': 87278757, '132_hero_id': 0, 'league_id': 17874}
game {'match_id': 8237714148, 'radiant_team_id': 9017006, 'radiant_name': 'NAVI Junior', 'dire_team_id': 9247798, 'dire_name': 'Passion UA', 'game_duration': nan, 'start_time': 1743506030.285119, 'radiant_win': -1, '0_account_id': nan, '0_hero_id': nan, '1_account_id': nan, '1_hero_id': nan, '2_account_id': nan, '2

In [8]:
def insert_to_set():
    games = retrieve_live_league_games()
    match_ids = [item['match_id'] for item in games]
    r.sadd('live_matches:0', *match_ids)
    print(r.smembers('live_matches:0'))

insert_to_set()
    

{'8237708401', '8237714148', '8237717392'}


In [ ]:
def insert_to_hset():
    games = retrieve_live_league_games()
    
    pipe = r.pipeline()
    for game in games:
        match_id = game['match_id']
        pipe.hset(f'live_match_details:{match_id}', mapping=game)
    
    pipe.execute()
    
    for key in r.scan_iter('live_match_details:*'):
        match_details = r.hgetall(key)
        print(f"{key}: {match_details}")
    
insert_to_hset()

live_match_details:8237708401: {'match_id': '8237708401', 'radiant_team_id': '8291895', 'radiant_name': 'Tundra Esports', 'dire_team_id': '1838315', 'dire_name': 'Team Secret', 'game_duration': '238.63336181640625', 'start_time': '1743506915.934136', 'radiant_win': '-1', '0_account_id': '86698277', '0_hero_id': '38', '1_account_id': '136829091', '1_hero_id': '103', '2_account_id': '127617979', '2_hero_id': '53', '3_account_id': '93618577', '3_hero_id': '74', '4_account_id': '103735745', '4_hero_id': '21', '128_account_id': '1171243748', '128_hero_id': '109', '129_account_id': '58513047', '129_hero_id': '17', '130_account_id': '374875067', '130_hero_id': '137', '131_account_id': '105045291', '131_hero_id': '34', '132_account_id': '87278757', '132_hero_id': '100', 'league_id': '17874'}
live_match_details:0: {'match_id': '8237717392', 'radiant_team_id': '9492830', 'radiant_name': 'Prodigy of God', 'dire_team_id': '8831040', 'dire_name': 'Ghost Sheep', 'game_duration': '10.36669921875', 's

In [ ]:
def find_completed_matches():
    r.delete('temp_curr_matches')
    games = retrieve_live_league_games()
    curr_poll_ids = [item['match_id'] for item in games]
    if curr_poll_ids:
        r.sadd('temp_curr_matches', *curr_poll_ids)
        
    completed = r.sdiff('live_matches:0', 'temp_curr_matches') # Present in first but not second
    return completed

completed = find_completed_matches()
completed

{'8237708401', '8237714148', '8237717392'}

## Testing Pipeline performance benchmark

In [25]:
import redis
import time

def test_without_pipeline(r, hash_name, items=10000):
    """Insert items into Redis hash without using pipeline"""
    start_time = time.time()
    
    for i in range(items):
        r.hset(hash_name, f"field_{i}", f"value_{i}")
    
    end_time = time.time()
    return end_time - start_time

def test_with_pipeline(r, hash_name, items=10000):
    """Insert items into Redis hash using pipeline"""
    start_time = time.time()
    
    pipe = r.pipeline()
    for i in range(items):
        pipe.hset(hash_name, f"field_{i}", f"value_{i}")
    pipe.execute()
    
    end_time = time.time()
    return end_time - start_time

# Connect to Redis
r = redis.Redis(host='localhost', port=6379, db=0)

# Clean up any existing test data
r.delete("test_hash_regular")
r.delete("test_hash_pipeline")

# Run tests
items = 10000
regular_time = test_without_pipeline(r, "test_hash_regular", items)
pipeline_time = test_with_pipeline(r, "test_hash_pipeline", items)

# Print results
print(f"Inserting {items} items without pipeline took: {regular_time:.2f} seconds")
print(f"Inserting {items} items with pipeline took: {pipeline_time:.2f} seconds")
print(f"Pipeline is {regular_time/pipeline_time:.1f}x faster")

Inserting 10000 items without pipeline took: 0.69 seconds
Inserting 10000 items with pipeline took: 0.15 seconds
Pipeline is 4.5x faster


In [27]:
import cProfile

cProfile.run("test_with_pipeline(r, 'test_hash_pipeline')")

         351096 function calls in 0.152 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.007    0.007    0.152    0.152 4112423164.py:14(test_with_pipeline)
        1    0.000    0.000    0.152    0.152 <string>:1(<module>)
        1    0.000    0.000    0.000    0.000 backoff.py:13(reset)
        1    0.000    0.000    0.000    0.000 client.py:1227(__init__)
        1    0.000    0.000    0.000    0.000 client.py:1243(__del__)
        3    0.000    0.000    0.000    0.000 client.py:1256(reset)
    10000    0.006    0.000    0.012    0.000 client.py:1296(execute_command)
    10000    0.004    0.000    0.005    0.000 client.py:1347(pipeline_execute_command)
        1    0.016    0.016    0.124    0.124 client.py:1362(_execute_transaction)
        1    0.001    0.001    0.001    0.001 client.py:1365(<listcomp>)
        1    0.002    0.002    0.003    0.003 client.py:1447(raise_first_error)
    10002    0.005    